In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import os

url = "https://books.toscrape.com/"

response = requests.get(url)
print("Status code:", response.status_code)

soup = BeautifulSoup(response.content, "html.parser")
print("Website title:", soup.title.text)

categories = {
    "Travel": url + "catalogue/category/books/travel_2/index.html",
    "Mystery": url + "catalogue/category/books/mystery_3/index.html",
    "Science Fiction": url + "catalogue/category/books/science-fiction_16/index.html",
    "Classics": url + "catalogue/category/books/classics_6/index.html",
    "Historical Fiction": url + "catalogue/category/books/historical-fiction_4/index.html"
}

all_books = []

for category, category_url in categories.items():

    response = requests.get(category_url)
    soup = BeautifulSoup(response.content, "html.parser")

    books = soup.find_all("article", class_="product_pod")

    for book in books:

        title = book.h3.a["title"]

        price = book.find(
            "p",
            class_="price_color"
        ).text.strip()

        rating = book.p["class"][1]

        availability = book.find(
            "p",
            class_="instock availability"
        ).text.strip()

        all_books.append({
            "title": title,
            "price": price,
            "rating": rating,
            "availability": availability,
            "category": category
        })

    print(category, ":", len(books), "books")

print("\nTotal books collected:", len(all_books))

if len(all_books) >= 60:
    print("60+ books requirement: PASSED")
else:
    print("60+ books requirement: FAILED")

df = pd.DataFrame(all_books)

print("\nDataset shape:", df.shape)

print("Total categories:", df["category"].nunique())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nFirst 5 rows:")
print(df.head())

df["price_gbp"] = (
    df["price"]
    .str.replace("£", "", regex=False)
    .astype(float)
)

gbp_to_inr = 128.28

df["price_inr"] = (df["price_gbp"] * gbp_to_inr).round(2)

rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["star_rating"] = df["rating"].map(rating_map)

df["in_stock"] = df["availability"].str.contains(
    "In stock",
    case=False,
    na=False
)

df_clean = df[["title","price_gbp","price_inr","star_rating","in_stock","category"]].copy()

print("\nCleaned dataset:")
print(df_clean.head())

print("\nData types:")
print(df_clean.dtypes)

df_clean.to_csv(
    "books_cleaned.csv",
    index=False
)

print("\nCSV file saved as books_cleaned.csv")

if os.path.exists("books_database.db"):
    os.remove("books_database.db")

conn = sqlite3.connect("books_database.db")

cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON")

print("SQLite database created.")

cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")

for category in df_clean["category"].unique():

    cursor.execute(
        """
        INSERT OR IGNORE INTO categories
        (category_name)
        VALUES (?)
        """,
        (category,)
    )

cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    star_rating INTEGER,
    in_stock BOOLEAN,
    category_id INTEGER,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")

for _, row in df_clean.iterrows():

    cursor.execute(
        """
        SELECT category_id
        FROM categories
        WHERE category_name = ?
        """,
        (row["category"],)
    )

    category_id = cursor.fetchone()[0]

    cursor.execute(
        """
        INSERT INTO books
        (
            title,
            price_gbp,
            price_inr,
            star_rating,
            in_stock,
            category_id
        )
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (
            row["title"],
            row["price_gbp"],
            row["price_inr"],
            row["star_rating"],
            row["in_stock"],
            category_id
        )
    )


conn.commit()

print("Books inserted into database.")

cursor.execute(
    "SELECT COUNT(*) FROM books"
)

book_count = cursor.fetchone()[0]

print("Books in SQLite:", book_count)

query1 = """
SELECT *
FROM books
"""

result1 = pd.read_sql_query(
    query1,
    conn
)

print("\nQuery 1 - SELECT:")
print(result1.head())

query2 = """
SELECT
    title,
    price_gbp,
    star_rating
FROM books
WHERE star_rating >= 4
"""

result2 = pd.read_sql_query(
    query2,
    conn
)

print("\nQuery 2 - WHERE:")
print(result2.head())

query3 = """
SELECT
    title,
    price_gbp
FROM books
ORDER BY price_gbp DESC
"""

result3 = pd.read_sql_query(
    query3,
    conn
)

print("\nQuery 3 - ORDER BY:")
print(result3.head())

query4 = """
SELECT
    title,
    price_gbp
FROM books
LIMIT 10
"""

result4 = pd.read_sql_query(
    query4,
    conn
)

print("\nQuery 4 - LIMIT:")
print(result4)

query5 = """
SELECT DISTINCT
    category_id
FROM books
"""

result5 = pd.read_sql_query(
    query5,
    conn
)

print("\nQuery 5 - DISTINCT:")
print(result5)

query6 = """
SELECT
    title,
    price_gbp,
    price_inr
FROM books
WHERE price_gbp BETWEEN 20 AND 40
"""

result6 = pd.read_sql_query(
    query6,
    conn
)

print("\nQuery 6 - BETWEEN:")
print(result6.head())

query7 = """
SELECT
    books.title,
    books.price_gbp,
    books.price_inr,
    books.star_rating,
    categories.category_name
FROM books
JOIN categories
ON books.category_id = categories.category_id
"""

result7 = pd.read_sql_query(
    query7,
    conn
)

print("\nQuery 7 - JOIN:")
print(result7.head())

books_df = pd.read_sql_query(
    "SELECT * FROM books",
    conn
)
categories_df = pd.read_sql_query(
    "SELECT * FROM categories",
    conn
)
merged_df = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)
print("\nPandas Merge:")
print(merged_df[["title","price_gbp","price_inr","star_rating","category_name"]].head())

print("Total books:",len(df_clean))
print("Total categories:",df_clean["category"].nunique())
print("Missing values:",df_clean.isnull().sum().sum())
print("Duplicate rows:",df_clean.duplicated().sum())
print("Books in SQLite:",book_count)

print("GBP column: Yes")
print("INR column: Yes")
print("SQLite database: Yes")
print("Primary key: Yes")
print("Foreign key: Yes")
print("SQL queries completed: 7")
print("SQL JOIN completed: Yes")
print("Pandas merge completed: Yes")

conn.close()

print("\nDatabase connection closed.")

Status code: 200
Website title: 
    All products | Books to Scrape - Sandbox

Travel : 11 books
Mystery : 20 books
Science Fiction : 16 books
Classics : 19 books
Historical Fiction : 20 books

Total books collected: 86
60+ books requirement: PASSED

Dataset shape: (86, 5)
Total categories: 5

Missing values:
title           0
price           0
rating          0
availability    0
category        0
dtype: int64

Duplicate rows: 0

First 5 rows:
                                               title   price rating  \
0                            It's Only the Himalayas  £45.17    Two   
1  Full Moon over Noah’s Ark: An Odyssey to Mount...  £49.43   Four   
2  See America: A Celebration of Our National Par...  £48.87  Three   
3  Vagabonding: An Uncommon Guide to the Art of L...  £36.94    Two   
4                               Under the Tuscan Sun  £37.33  Three   

  availability category  
0     In stock   Travel  
1     In stock   Travel  
2     In stock   Travel  
3     In stock   Trav